In [1]:
import os
os.chdir('/kaggle/working')
!rm -rf /kaggle/working/repo
!pip uninstall ultralytics -y -q
!git clone https://github.com/AadeeshRS/road-damage-detection-thesis.git /kaggle/working/repo
!pip install -e /kaggle/working/repo/ultralytics -q
!pip install sahi -q
print("Setup done!")


Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 1504, done.
remote: Counting objects: 100% (1108/1108), done.
remote: Compressing objects: 100% (858/858), done.
remote: Total 1504 (delta 278), reused 1055 (delta 234), pack-reused 396 (from 3)
Receiving objects: 100% (1504/1504), 478.24 MiB | 40.69 MiB/s, done.
Resolving deltas: 100% (339/339), done.
Updating files: 100% (1190/1190), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.6/148.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.0 MB/s eta 0:00:00
Setup done!


In [2]:
import sys
for key in list(sys.modules.keys()):
    if 'ultralytics' in key:
        del sys.modules[key]
sys.path.insert(0, '/kaggle/working/repo/ultralytics')

import torch
from ultralytics import YOLO
print("CUDA:", torch.cuda.is_available())

model = YOLO('/kaggle/working/repo/ultralytics/ultralytics/cfg/models/v8/yolov8m-full-hybrid.yaml')
print("SUCCESS! Params:", f"{sum(p.numel() for p in model.model.parameters()):,}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
CUDA: True
LEGACY = True
SUCCESS! Params: 23,320,583


In [3]:
yaml_content = """
path: /kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT

train: train/images
val: val/images
test: test/images

names:
  0: longitudinal_crack
  1: transverse_crack
  2: alligator_crack
  3: other_corruption
  4: pothole
"""
with open("/kaggle/working/dataset.yaml", "w") as f:
    f.write(yaml_content)
print("dataset.yaml created")


dataset.yaml created


In [4]:
import os
os.chdir('/kaggle/working/repo')
!git pull origin main


remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 7 (delta 3), reused 7 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 183.69 KiB | 1.77 MiB/s, done.
From https://github.com/AadeeshRS/road-damage-detection-thesis
 * branch            main       -> FETCH_HEAD
   7386471..28183f3  main       -> origin/main
Updating 7386471..28183f3
Fast-forward
 ultralytics/ultralytics/assets/bus.jpg    | Bin 0 -> 137419 bytes
 ultralytics/ultralytics/assets/zidane.jpg | Bin 0 -> 50427 bytes
 2 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ultralytics/ultralytics/assets/bus.jpg
 create mode 100644 ultralytics/ultralytics/assets/zidane.jpg


In [5]:
model = YOLO('/kaggle/working/repo/ultralytics/ultralytics/cfg/models/v8/yolov8m-dat.yaml')
print(f"Total Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

LEGACY = True
Total Parameters: 22,727,351


In [ ]:
results = model.train(
    data="/kaggle/working/dataset.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    workers=4,
    project="thesis_experiments",
    name="ablation_dat_only"
)

New https://pypi.org/project/ultralytics/8.4.103 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/repo/ultralytics/ultralytics/

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30      6.89G      3.352      4.403       3.34         10        640: 100% ━━━━━━━━━━━━ 1680/1680 1.5it/s 18:250.8ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 180/180 1.8it/s 1:400.5ss
                   all       5758       9740      0.283     0.0214     0.0181    0.00511

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30      7.07G      2.778       3.72      2.565         52        640: 0% ──────────── 0/1680  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/30      7.09G      2.357      3.297      2.233         21        640: 100% ━━━━━━━━━━━━ 1680/1680 1.7it/s 16:430.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 180/180 1.9it/s 1:320.5sss
                   all       5758       9740      0.207      0.193      0.111     0.0395

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      7.07G      2.219      2.897      2.033         49        640: 0% ──────────── 0/1680  0.6s

...

      16/30      7.09G      1.648      1.863      1.587         18        640: 100% ━━━━━━━━━━━━ 1680/1680 1.8it/s 15:570.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 180/180 2.0it/s 1:320.5sss
                   all       5758       9740      0.563      0.517      0.524      0.273

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      17/30      7.09G      1.686      1.808      1.506         48        640: 0% ──────────── 0/1680  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      17/30      7.11G      1.628       1.83      1.572          8        640: 100% ━━━━━━━━━━━━ 1680/1680 1.8it/s 15:560.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 180/180 2.0it/s 1:310.5sss
                   all       5758       9740      0.576      0.521      0.536      0.284

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      18/30      7.07G      1.598      1.606       1.54         38        640: 0% ──────────── 0/1680  0.5s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      18/30      7.08G      1.627      1.807      1.568         36        640: 19% ━━────────── 321/1680 1.8it/s 3:04<12:27

In [1]:
import os
import shutil

# 1. MOVE THE RUNS FOLDER TO SAFETY!
old_runs_path = '/kaggle/working/repo/runs'
safe_runs_path = '/kaggle/working/runs'

if os.path.exists(old_runs_path):
    # If a runs folder already exists in the safe spot, merge/overwrite safely
    if os.path.exists(safe_runs_path):
        !cp -r /kaggle/working/repo/runs/* /kaggle/working/runs/
        print("Copied runs to safety!")
    else:
        shutil.move(old_runs_path, safe_runs_path)
        print("Moved runs folder to safety!")
else:
    print("No runs folder found in repo (it might already be in the safe spot).")

# 2. Now it is safe to delete and reinstall the repo
os.chdir('/kaggle/working')
!rm -rf /kaggle/working/repo
!git clone https://github.com/AadeeshRS/road-damage-detection-thesis.git /kaggle/working/repo
!pip uninstall ultralytics -y -q
!pip install -e /kaggle/working/repo/ultralytics -q

# 3. Clear Cache
import sys
for key in list(sys.modules.keys()):
    if 'ultralytics' in key:
        del sys.modules[key]
sys.path.insert(0, '/kaggle/working/repo/ultralytics')

print("Custom Hybrid repo setup complete & weights are safe!")


Moved runs folder to safety!
Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 1511, done.
remote: Counting objects: 100% (1115/1115), done.
remote: Compressing objects: 100% (862/862), done.
remote: Total 1511 (delta 281), reused 1062 (delta 237), pack-reused 396 (from 3)
Receiving objects: 100% (1511/1511), 478.42 MiB | 43.08 MiB/s, done.
Resolving deltas: 100% (342/342), done.
Updating files: 100% (1192/1192), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
Custom Hybrid repo setup complete & weights are safe!


In [3]:
from ultralytics import YOLO

# Notice the path now points to the safe `/kaggle/working/runs/` folder!
last_weights_path = "/kaggle/working/runs/detect/thesis_experiments/ablation_dat_only/weights/last.pt"

# Load the model
model = YOLO(last_weights_path)
print("Found interrupted model!")

# Resume training
results = model.train(resume=True)


Found interrupted model!
New https://pypi.org/project/ultralytics/8.4.104 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/runs

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      19/30      7.05G      1.596      1.759      1.545         10        640: 100% ━━━━━━━━━━━━ 1680/1680 1.6it/s 17:030.8ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 180/180 1.8it/s 1:410.5ss
                   all       5758       9740      0.601      0.529      0.555      0.295

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      20/30      7.14G      1.593      1.748      1.541         21        640: 100% ━━━━━━━━━━━━ 1680/1680 1.7it/s 16:550.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 180/180 1.9it/s 1:340.5sss
                   all       5758       9740      0.604      0.538      0.561      0.298
Closing dataloader mosaic
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

      28/30      7.14G      1.439      1.355      1.469         16        640: 100% ━━━━━━━━━━━━ 1680/1680 1.7it/s 16:490.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 180/180 1.9it/s 1:350.5sss
                   all       5758       9740      0.642      0.563      0.604      0.326

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/30      7.14G      1.423      1.328      1.462         11        640: 100% ━━━━━━━━━━━━ 1680/1680 1.7it/s 16:480.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 180/180 1.9it/s 1:350.5sss
                   all       5758       9740      0.641      0.569      0.604      0.326

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30      7.14G      1.405      1.295      1.449          8        640: 100% ━━━━━━━━━━━━ 1680/1680 1.7it/s 16:490.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 180/180 1.8it/s 1:420.5ss
                   all       5758       9740      0.646       0.57      0.607      0.326

12 epochs completed in 3.694 hours.
Optimizer stripped from /kaggle/working/repo/runs/detect/thesis_experiments/ablation_dat_only/weights/last.pt, 45.8MB
Optimizer stripped from /kaggle/working/repo/runs/detect/thesis_experiments/ablation_dat_only/weights/best.pt, 45.8MB

Validating /kaggle/working/repo/runs/detect/thesis_experiments/ablation_dat_only/weights/best.pt...
Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8m-dat summary (fused): 110 layers, 22,712,327 parameters, 0 gradients, 65.0 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━

In [1]:
!cd /kaggle/working/repo/runs/detect/thesis_experiments && zip -r /kaggle/working/ablation_dat_only_results.zip ablation_dat_only

  adding: ablation_dat_only/ (stored 0%)
  adding: ablation_dat_only/args.yaml (deflated 56%)
  adding: ablation_dat_only/BoxPR_curve.png (deflated 7%)
  adding: ablation_dat_only/val_batch1_labels.jpg (deflated 8%)
  adding: ablation_dat_only/confusion_matrix.png (deflated 20%)
  adding: ablation_dat_only/BoxF1_curve.png (deflated 9%)
  adding: ablation_dat_only/val_batch1_pred.jpg (deflated 8%)
  adding: ablation_dat_only/results.png (deflated 7%)
  adding: ablation_dat_only/val_batch2_labels.jpg (deflated 10%)
  adding: ablation_dat_only/BoxR_curve.png (deflated 8%)
  adding: ablation_dat_only/val_batch2_pred.jpg (deflated 10%)
  adding: ablation_dat_only/val_batch0_labels.jpg (deflated 9%)
  adding: ablation_dat_only/val_batch0_pred.jpg (deflated 9%)
  adding: ablation_dat_only/train_batch33602.jpg (deflated 7%)
  adding: ablation_dat_only/weights/ (stored 0%)
  adding: ablation_dat_only/weights/last.pt (deflated 8%)
  adding: ablation_dat_only/weights/best.pt (deflated 8%)
  addin